# Gerar CSV de métricas IMDB-WIKI com SAM 3





## 1. Instalação das dependências

As células abaixo preservam a estrutura de instalação separada. Em RunPod, execute-as na ordem antes de inicializar o SAM 3.


In [ ]:
%pip install -q numpy pandas pillow scipy matplotlib seaborn scikit-learn tqdm torch torchvision requests

In [ ]:
!pip install -q einops


In [ ]:
!pip install -q decord


In [ ]:
!pip install -q pycocotools


In [ ]:
!pip install -q huggingface_hub


In [ ]:
!pip install -q pyarrow


## 2. Instalação idempotente do SAM 3

O código do SAM 3 é fixado em um commit específico. Se `/workspace/sam3` já existir, o notebook usa a pasta existente, faz `git fetch` e força o checkout do commit fixado.


In [ ]:
from pathlib import Path
import subprocess

SAM3_DIR = Path("/workspace/sam3")
SAM3_GIT_REPO = "https://github.com/facebookresearch/sam3.git"
SAM3_CODE_COMMIT = "5dd401d1c5c1d5c3eedff06d41b77af824517619"
SAM3_MODEL_REPO = "facebook/sam3"
SAM3_CHECKPOINT = "sam3.pt"

if not SAM3_DIR.exists():
    SAM3_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", SAM3_GIT_REPO, str(SAM3_DIR)], check=True)
else:
    print(f"Usando repositório SAM 3 existente em {SAM3_DIR}")

subprocess.run(["git", "-C", str(SAM3_DIR), "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", str(SAM3_DIR), "checkout", SAM3_CODE_COMMIT], check=True)

current_commit = subprocess.check_output(
    ["git", "-C", str(SAM3_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

assert current_commit == SAM3_CODE_COMMIT, f"Commit SAM 3 inesperado: {current_commit}"
print(f"SAM 3 fixado no commit: {current_commit}")


In [ ]:
%pip install -q -e /workspace/sam3


In [ ]:
from huggingface_hub import login

login()

## 3. Importações, autenticação e configuração



In [ ]:
%cd /workspace/sam3

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import requests
import pandas as pd
from PIL import Image, UnidentifiedImageError
from scipy.io import loadmat
from scipy.ndimage import binary_closing, binary_opening

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda iterable, **_: iterable

from huggingface_hub import hf_hub_download, login, repo_info

import torch

try:
    import torchvision
except ImportError:
    torchvision = None

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HF_TOKEN encontrado: login realizado no Hugging Face.")
else:
    print(
        "HF_TOKEN não encontrado. Configure a variável de ambiente antes de inicializar o SAM 3 "
        "se o checkpoint exigir autorização previamente aceita na página do Hugging Face."
    )


In [ ]:
DATASET_ROOT = Path("/workspace/datasets/imdb_wiki")
DOWNLOAD_ROOT = DATASET_ROOT / "downloads"
EXTRACT_ROOT = DATASET_ROOT / "extracted"
OUTPUT_ROOT = Path("/workspace/outputs_imdb_shpm")

DATASET_KIND = "imdb"  # valores permitidos: "imdb" ou "wiki"
RUN_MODE = "sample"    # valores permitidos: "smoke_test", "sample" ou "full"

MAX_IMAGES_PER_GENDER = {
    "smoke_test": 10,
    "sample": 1000,
    "full": None,
}[RUN_MODE]

RANDOM_SEED = 42
SCORE_THRESHOLD = 0.5
MIN_FACE_AREA = 100

SAVE_QC_IMAGES = True
QC_IMAGES_PER_GENDER = 25
QC_FAILURE_IMAGES = 25
CHECKPOINT_EVERY = 250

MIN_METADATA_FACE_SCORE = 1.0
REQUIRE_SINGLE_FACE = True
MIN_IMAGE_WIDTH = 1
MIN_IMAGE_HEIGHT = 1
VALIDATE_IMAGE_FILES_BEFORE_SELECTION = True

IMDB_URL = "https://data.vision.ee.ethz.ch/cvl/rrothe/imdb-wiki/static/imdb_crop.tar"
WIKI_URL = "https://data.vision.ee.ethz.ch/cvl/rrothe/imdb-wiki/static/wiki_crop.tar"

if DATASET_KIND not in {"imdb", "wiki"}:
    raise ValueError("DATASET_KIND deve ser 'imdb' ou 'wiki'.")
if RUN_MODE not in {"smoke_test", "sample", "full"}:
    raise ValueError("RUN_MODE deve ser 'smoke_test', 'sample' ou 'full'.")

DATASET_URL = os.getenv("IMDB_ARCHIVE_URL", IMDB_URL) if DATASET_KIND == "imdb" else WIKI_URL
ARCHIVE_NAME = f"{DATASET_KIND}_crop.tar"
ARCHIVE_PATH = DOWNLOAD_ROOT / ARCHIVE_NAME
CROP_DIR_NAME = f"{DATASET_KIND}_crop"
MAT_FILE_NAME = f"{DATASET_KIND}.mat"
FILE_PREFIX = DATASET_KIND

CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
QC_ROOT = OUTPUT_ROOT / "qc"
SUMMARY_ROOT = OUTPUT_ROOT / "summary"

for folder in [DOWNLOAD_ROOT, EXTRACT_ROOT, OUTPUT_ROOT, CHECKPOINT_ROOT, QC_ROOT, SUMMARY_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

SAM3_PROMPTS = {
    "face": "face",
    "nose": "nose",
    "mouth": "mouth",
    "l_eye": "left eye",
    "r_eye": "right eye",
    "hair": "hair",
}

parts = [
    "face",
    "nose",
    "mouth",
    "l_eye",
    "r_eye",
    "hair",
]

feature_names = [
    f"{part_i}_vs_{part_j}"
    for part_i in parts
    for part_j in parts
    if part_i != part_j
]

assert len(feature_names) == 30

MASK_COLORS = {
    "face": (250, 184, 132),
    "nose": (141, 211, 199),
    "mouth": (255, 128, 128),
    "l_eye": (128, 177, 211),
    "r_eye": (188, 128, 189),
    "hair": (190, 174, 212),
}

np.random.seed(RANDOM_SEED)

print(f"Dataset: {DATASET_KIND}")
print(f"Modo: {RUN_MODE} | máximo por gênero: {MAX_IMAGES_PER_GENDER}")
print(f"Saídas: {OUTPUT_ROOT}")
print(feature_names)


## 4. Download e extração automática do IMDB-WIKI



In [ ]:
def human_size(num_bytes: int | float) -> str:
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024:
            return f"{value:.2f} {unit}"
        value /= 1024
    return f"{value:.2f} PB"


def disk_report(path: Path) -> dict[str, str]:
    usage = shutil.disk_usage(path)
    report = {
        "total": human_size(usage.total),
        "used": human_size(usage.used),
        "free": human_size(usage.free),
    }
    print(
        f"Disco em {path}: total={report['total']} | "
        f"usado={report['used']} | livre={report['free']}"
    )
    return report


def archive_looks_complete(archive_path: Path) -> bool:
    """Verifica silenciosamente se o arquivo TAR pode ser listado por completo."""
    if not archive_path.exists() or archive_path.stat().st_size == 0:
        return False

    result = subprocess.run(
        ["tar", "-tf", str(archive_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    return result.returncode == 0


def download_archive(
    url: str,
    archive_path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> None:
    """
    Baixa o arquivo com uma única barra de progresso.

    Se houver um arquivo parcial, tenta continuar o download usando HTTP Range.
    Caso o servidor não aceite retomada, o download é reiniciado automaticamente.
    """
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    disk_report(archive_path.parent)

    existing_size = archive_path.stat().st_size if archive_path.exists() else 0

    if existing_size > 0:
        print(f"Arquivo existente: {archive_path} ({human_size(existing_size)})")
        print("Verificando se o TAR já está completo...")

        if archive_looks_complete(archive_path):
            print("Arquivo TAR completo; download não será repetido.")
            return

        print("Arquivo parcial detectado; tentando continuar o download.")

    request_headers = {
        "User-Agent": "Mozilla/5.0",
    }

    if existing_size > 0:
        request_headers["Range"] = f"bytes={existing_size}-"

    response = requests.get(
        url,
        stream=True,
        headers=request_headers,
        timeout=(30, 300),
        allow_redirects=True,
    )

    # HTTP 416 pode indicar que o arquivo local já possui o tamanho remoto.
    if response.status_code == 416:
        response.close()

        if archive_looks_complete(archive_path):
            print("O arquivo local já está completo.")
            return

        print("Não foi possível retomar o arquivo parcial; reiniciando o download.")
        archive_path.unlink(missing_ok=True)
        existing_size = 0

        response = requests.get(
            url,
            stream=True,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=(30, 300),
            allow_redirects=True,
        )

    response.raise_for_status()

    if existing_size > 0 and response.status_code == 206:
        file_mode = "ab"
        initial_size = existing_size
    else:
        # O servidor ignorou o cabeçalho Range ou não havia arquivo parcial.
        file_mode = "wb"
        initial_size = 0

    content_range = response.headers.get("Content-Range", "")
    content_length = response.headers.get("Content-Length")

    if "/" in content_range:
        total_size_text = content_range.rsplit("/", 1)[-1]
        total_size = int(total_size_text) if total_size_text.isdigit() else None
    elif content_length and content_length.isdigit():
        received_size = int(content_length)
        total_size = received_size + initial_size
    else:
        total_size = None

    with response:
        with open(archive_path, file_mode) as file_handle:
            with tqdm(
                total=total_size,
                initial=initial_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc=f"Baixando {archive_path.name}",
                dynamic_ncols=True,
                mininterval=0.5,
            ) as progress:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    if not chunk:
                        continue
                    file_handle.write(chunk)
                    progress.update(len(chunk))

    print("Validando o arquivo TAR baixado...")

    if not archive_looks_complete(archive_path):
        raise RuntimeError(
            "O download terminou, mas o arquivo TAR não pôde ser listado. "
            f"O arquivo parcial foi preservado em {archive_path} para permitir retomada."
        )

    print(f"Download concluído: {archive_path} ({human_size(archive_path.stat().st_size)})")


def find_dataset_paths(dataset_kind: str, extract_root: Path) -> tuple[Path, Path]:
    crop_dir_name = f"{dataset_kind}_crop"
    mat_file_name = f"{dataset_kind}.mat"

    preferred_crop_dir = extract_root / crop_dir_name
    preferred_mat = preferred_crop_dir / mat_file_name

    if preferred_crop_dir.exists() and preferred_mat.exists():
        return preferred_mat, preferred_crop_dir

    mat_candidates = sorted(extract_root.rglob(mat_file_name))
    crop_candidates = [
        path for path in extract_root.rglob(crop_dir_name)
        if path.is_dir()
    ]

    if not mat_candidates or not crop_candidates:
        raise FileNotFoundError(
            f"Extração inválida para {dataset_kind}: "
            f"esperava {mat_file_name} e a pasta {crop_dir_name}."
        )

    return mat_candidates[0], crop_candidates[0]


def crop_dir_has_images(crop_dir: Path) -> bool:
    image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

    for path in crop_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in image_extensions:
            return True

    return False


def extraction_marker_path(dataset_kind: str, extract_root: Path) -> Path:
    return extract_root / f".{dataset_kind}_extraction_complete"


def extraction_is_valid(dataset_kind: str, extract_root: Path) -> bool:
    marker_path = extraction_marker_path(dataset_kind, extract_root)

    if not marker_path.exists():
        return False

    try:
        mat_path, crop_dir = find_dataset_paths(dataset_kind, extract_root)
        has_images = crop_dir_has_images(crop_dir)

        if not has_images:
            return False

        print(f"MAT encontrado: {mat_path}")
        print(f"Pasta de imagens: {crop_dir}")
        return True

    except FileNotFoundError:
        return False


def extract_archive(
    archive_path: Path,
    extract_root: Path,
    dataset_kind: str,
) -> tuple[Path, Path]:
    """
    Extrai o dataset sem imprimir o nome de cada arquivo.

    --no-same-owner evita os erros de UID/GID observados em sistemas de arquivos
    usados por ambientes como o RunPod.
    """
    if extraction_is_valid(dataset_kind, extract_root):
        print("Extração já validada; o TAR não será extraído novamente.")
        return find_dataset_paths(dataset_kind, extract_root)

    extract_root.mkdir(parents=True, exist_ok=True)
    marker_path = extraction_marker_path(dataset_kind, extract_root)
    marker_path.unlink(missing_ok=True)

    print(f"Extraindo {archive_path.name}. Esta etapa pode levar alguns minutos...")

    command = [
        "tar",
        "--no-same-owner",
        "--no-same-permissions",
        "-xf",
        str(archive_path),
        "-C",
        str(extract_root),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        error_lines = [
            line.strip()
            for line in result.stderr.splitlines()
            if line.strip()
        ]
        error_summary = "\n".join(error_lines[-10:])

        raise RuntimeError(
            "A extração do dataset falhou.\n"
            f"Comando: {' '.join(command)}\n"
            f"Últimas mensagens do TAR:\n{error_summary}"
        )

    try:
        mat_path, crop_dir = find_dataset_paths(dataset_kind, extract_root)
    except FileNotFoundError as exc:
        raise RuntimeError(
            "A extração terminou, mas os arquivos esperados não foram encontrados."
        ) from exc

    if not crop_dir_has_images(crop_dir):
        raise RuntimeError(
            f"A pasta extraída não contém imagens reconhecidas: {crop_dir}"
        )

    marker_path.write_text(
        datetime.now(timezone.utc).isoformat(),
        encoding="utf-8",
    )

    print("Extração concluída e validada.")
    return mat_path, crop_dir


download_archive(DATASET_URL, ARCHIVE_PATH)
MAT_PATH, IMAGE_ROOT = extract_archive(
    ARCHIVE_PATH,
    EXTRACT_ROOT,
    DATASET_KIND,
)

print(f"MAT_PATH = {MAT_PATH}")
print(f"IMAGE_ROOT = {IMAGE_ROOT}")

## 5. Leitura e filtragem dos metadados



In [ ]:
def unwrap_matlab_value(value: Any) -> Any:
    current = value
    while isinstance(current, np.ndarray) and current.dtype == object and current.size == 1:
        current = current.item()
    return current


def matlab_to_string(value: Any) -> str:
    value = unwrap_matlab_value(value)

    if isinstance(value, bytes):
        return value.decode("utf-8", errors="ignore")

    if isinstance(value, str):
        return value

    arr = np.asarray(value).squeeze()

    if arr.size == 0:
        return ""

    if arr.dtype.kind in {"U", "S"}:
        if arr.ndim == 0:
            item = arr.item()
            return item.decode("utf-8", errors="ignore") if isinstance(item, bytes) else str(item)
        return "".join(str(item) for item in arr.tolist())

    if arr.dtype == object:
        pieces = [matlab_to_string(item) for item in arr.reshape(-1)]
        return "".join(pieces).strip()

    if arr.size == 1:
        return str(arr.item())

    return str(arr.tolist())


def matlab_to_float(value: Any) -> float:
    value = unwrap_matlab_value(value)
    arr = np.asarray(value).squeeze()
    if arr.size == 0:
        return np.nan
    try:
        return float(arr.reshape(-1)[0])
    except Exception:
        return np.nan


def matlab_to_int_or_none(value: Any) -> int | None:
    number = matlab_to_float(value)
    if not np.isfinite(number):
        return None
    return int(number)


def matlab_to_float_list(value: Any) -> list[float]:
    value = unwrap_matlab_value(value)
    arr = np.asarray(value).squeeze()
    if arr.size == 0:
        return []
    try:
        return [float(item) for item in arr.reshape(-1)]
    except Exception:
        return []


def get_mat_struct_field(mat_struct: Any, field_name: str) -> Any:
    current = unwrap_matlab_value(mat_struct)
    if isinstance(current, np.ndarray) and current.dtype.names:
        return current[field_name][0, 0]
    if hasattr(current, field_name):
        return getattr(current, field_name)
    raise KeyError(field_name)


def get_optional_mat_struct_field(mat_struct: Any, field_name: str, n: int) -> list[Any]:
    try:
        field = get_mat_struct_field(mat_struct, field_name)
    except KeyError:
        return [np.nan] * n

    arr = np.asarray(field).squeeze()

    if field_name == "face_location" and arr.dtype != object and arr.ndim == 2:
        if arr.shape[0] == 4:
            return [arr[:, index] for index in range(arr.shape[1])]
        if arr.shape[1] == 4:
            return [arr[index, :] for index in range(arr.shape[0])]

    if arr.dtype == object:
        return list(arr.reshape(-1))
    return list(arr.reshape(-1))


def load_imdb_wiki_metadata(mat_path: Path, dataset_kind: str, image_root: Path) -> pd.DataFrame:
    mat = loadmat(mat_path, squeeze_me=False, struct_as_record=False)
    if dataset_kind not in mat:
        raise KeyError(f"Campo {dataset_kind!r} não encontrado em {mat_path}")

    mat_struct = mat[dataset_kind]
    full_paths = get_optional_mat_struct_field(mat_struct, "full_path", 0)
    n = len(full_paths)

    fields = {
        "gender": get_optional_mat_struct_field(mat_struct, "gender", n),
        "dob": get_optional_mat_struct_field(mat_struct, "dob", n),
        "photo_taken": get_optional_mat_struct_field(mat_struct, "photo_taken", n),
        "name": get_optional_mat_struct_field(mat_struct, "name", n),
        "face_location": get_optional_mat_struct_field(mat_struct, "face_location", n),
        "face_score": get_optional_mat_struct_field(mat_struct, "face_score", n),
        "second_face_score": get_optional_mat_struct_field(mat_struct, "second_face_score", n),
        "celeb_id": get_optional_mat_struct_field(mat_struct, "celeb_id", n),
    }

    rows = []
    for i in range(n):
        relative_path = matlab_to_string(full_paths[i]).replace("\\", "/").strip("/")
        image_path = image_root / relative_path

        raw_gender = matlab_to_float(fields["gender"][i])
        if raw_gender == 0:
            gender = "Female"
        elif raw_gender == 1:
            gender = "Male"
        else:
            gender = None

        rows.append({
            "dataset": dataset_kind,
            "filename": Path(relative_path).name,
            "relative_path": relative_path,
            "image_path": str(image_path),
            "gender": gender,
            "celeb_name": matlab_to_string(fields["name"][i]),
            "celeb_id": matlab_to_int_or_none(fields["celeb_id"][i]),
            "photo_taken": matlab_to_int_or_none(fields["photo_taken"][i]),
            "dob": matlab_to_float(fields["dob"][i]),
            "face_location": json.dumps(matlab_to_float_list(fields["face_location"][i])),
            "metadata_face_score": matlab_to_float(fields["face_score"][i]),
            "metadata_second_face_score": matlab_to_float(fields["second_face_score"][i]),
        })

    return pd.DataFrame(rows)


def image_is_readable(path: str | Path) -> tuple[bool, str, int | None, int | None]:
    try:
        with Image.open(path) as img:
            width, height = img.size
            if width < MIN_IMAGE_WIDTH or height < MIN_IMAGE_HEIGHT:
                return False, f"dimensão inválida: {width}x{height}", width, height
            img.verify()
        return True, "", width, height
    except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError) as exc:
        return False, f"{type(exc).__name__}: {exc}", None, None


def filter_metadata(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    report: dict[str, int] = {"total_raw": len(raw_df)}
    df = raw_df.copy()

    unknown_gender = ~df["gender"].isin(["Female", "Male"])
    report["unknown_gender"] = int(unknown_gender.sum())
    df = df.loc[~unknown_gender].copy()

    duplicated = df.duplicated("relative_path", keep="first")
    report["duplicated_relative_path"] = int(duplicated.sum())
    df = df.loc[~duplicated].copy()

    exists_mask = df["image_path"].map(lambda path: Path(path).exists())
    report["missing_path"] = int((~exists_mask).sum())
    df = df.loc[exists_mask].copy()

    face_score = pd.to_numeric(df["metadata_face_score"], errors="coerce")
    invalid_face_score = (~np.isfinite(face_score)) | (face_score < MIN_METADATA_FACE_SCORE)
    report["invalid_metadata_face_score"] = int(invalid_face_score.sum())
    df = df.loc[~invalid_face_score].copy()

    second_face_score = pd.to_numeric(df["metadata_second_face_score"], errors="coerce")
    if REQUIRE_SINGLE_FACE:
        multiple_faces = np.isfinite(second_face_score)
        report["multiple_faces_by_second_face_score"] = int(multiple_faces.sum())
        df = df.loc[~multiple_faces].copy()
    else:
        report["multiple_faces_by_second_face_score"] = 0

    if VALIDATE_IMAGE_FILES_BEFORE_SELECTION:
        readability = [
            image_is_readable(path)
            for path in tqdm(df["image_path"], desc="Validando arquivos de imagem")
        ]
        readable = np.array([item[0] for item in readability], dtype=bool)
        errors = [item[1] for item in readability]
        widths = [item[2] for item in readability]
        heights = [item[3] for item in readability]
        df["image_width"] = widths
        df["image_height"] = heights
        df["precheck_error"] = errors
        report["corrupted_or_invalid_image"] = int((~readable).sum())
        df = df.loc[readable].copy()
    else:
        report["corrupted_or_invalid_image"] = 0
        df["image_width"] = np.nan
        df["image_height"] = np.nan
        df["precheck_error"] = ""

    report["total_after_filters"] = len(df)
    report["female_after_filters"] = int((df["gender"] == "Female").sum())
    report["male_after_filters"] = int((df["gender"] == "Male").sum())

    return df.reset_index(drop=True), report


raw_metadata_df = load_imdb_wiki_metadata(MAT_PATH, DATASET_KIND, IMAGE_ROOT)
metadata_df, metadata_filter_report = filter_metadata(raw_metadata_df)

print("Relatório de metadados:")
print(json.dumps(metadata_filter_report, indent=2, ensure_ascii=False))
print(metadata_df["gender"].value_counts(dropna=False))
display(metadata_df.head())


In [ ]:
def deterministic_metadata_selection(df: pd.DataFrame) -> pd.DataFrame:
    selected_parts = []
    if MAX_IMAGES_PER_GENDER is None:
        selected = df.sort_values(["gender", "relative_path"]).reset_index(drop=True)
        print(f"RUN_MODE full: selecionadas {len(selected)} imagens válidas.")
        return selected

    counts = df["gender"].value_counts()
    per_gender = min(MAX_IMAGES_PER_GENDER, int(counts.get("Female", 0)), int(counts.get("Male", 0)))
    if per_gender <= 0:
        raise ValueError("Não há amostras suficientes de Female e Male após os filtros.")

    for gender in ["Female", "Male"]:
        group = df.loc[df["gender"] == gender].sort_values("relative_path")
        sampled = group.sample(n=per_gender, random_state=RANDOM_SEED)
        selected_parts.append(sampled)

    selected = pd.concat(selected_parts, axis=0).sort_values(["gender", "relative_path"]).reset_index(drop=True)
    print(f"Selecionadas {per_gender} imagens por gênero para RUN_MODE={RUN_MODE}.")
    return selected


selected_metadata_df = deterministic_metadata_selection(metadata_df)

print("Imagens selecionadas para processamento:")
print(selected_metadata_df["gender"].value_counts())
display(selected_metadata_df[["dataset", "relative_path", "gender", "metadata_face_score", "metadata_second_face_score"]].head())


## 6. Inicialização do SAM 3

Usa `facebook/sam3` com checkpoint `sam3.pt`. O notebook registra o commit do código, a revisão do Hugging Face quando disponível e o dtype usado pelo autocast.


In [ ]:
HF_REVISION = None
SAM3_CHECKPOINT_PATH = None
AUTOCast_DTYPE_USED = "cpu"


def get_sam3_code_commit() -> str:
    return subprocess.check_output(
        ["git", "-C", str(SAM3_DIR), "rev-parse", "HEAD"],
        text=True,
    ).strip()


def resolve_hf_revision(repo_id: str) -> str | None:
    try:
        info = repo_info(repo_id=repo_id, token=os.getenv("HF_TOKEN"))
        return getattr(info, "sha", None)
    except Exception as exc:
        print(
            "Não foi possível consultar a revisão do Hugging Face. "
            "Se o modelo exigir autorização, aceite os termos na página do modelo e configure HF_TOKEN. "
            f"Erro: {type(exc).__name__}: {exc}"
        )
        return None


def download_sam3_checkpoint() -> Path:
    global HF_REVISION, SAM3_CHECKPOINT_PATH
    HF_REVISION = resolve_hf_revision(SAM3_MODEL_REPO)
    try:
        checkpoint = hf_hub_download(
            repo_id=SAM3_MODEL_REPO,
            filename=SAM3_CHECKPOINT,
            token=os.getenv("HF_TOKEN"),
        )
    except Exception as exc:
        raise RuntimeError(
            "Falha ao baixar o checkpoint facebook/sam3/sam3.pt. "
            "Verifique conexão, HF_TOKEN e autorização do modelo no Hugging Face."
        ) from exc
    SAM3_CHECKPOINT_PATH = Path(checkpoint)
    print(f"Checkpoint SAM 3: {SAM3_CHECKPOINT_PATH}")
    return SAM3_CHECKPOINT_PATH


def initialize_sam3() -> tuple[Any, Any, str]:
    global AUTOCast_DTYPE_USED

    current_commit = get_sam3_code_commit()
    assert current_commit == SAM3_CODE_COMMIT, f"Commit SAM 3 inesperado: {current_commit}"
    assert SAM3_CHECKPOINT == "sam3.pt"

    checkpoint_path = download_sam3_checkpoint()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    build_attempts = [
        {"checkpoint_path": str(checkpoint_path)},
        {"ckpt_path": str(checkpoint_path)},
        {"checkpoint": str(checkpoint_path)},
        {},
    ]

    last_error = None
    model = None
    for kwargs in build_attempts:
        try:
            model = build_sam3_image_model(**kwargs)
            print(f"build_sam3_image_model executado com argumentos: {kwargs}")
            break
        except TypeError as exc:
            last_error = exc

    if model is None:
        raise RuntimeError(f"Não foi possível construir o modelo SAM 3: {last_error}")

    if hasattr(model, "to"):
        model = model.to(device)
    model.eval()

    if device == "cuda":
        supports_bfloat16 = getattr(torch.cuda, "is_bf16_supported", lambda: False)()
        AUTOCast_DTYPE_USED = "bfloat16" if supports_bfloat16 else "float16"
    else:
        AUTOCast_DTYPE_USED = "cpu"

    processor = Sam3Processor(model)
    print(f"SAM 3 inicializado em {device}; autocast={AUTOCast_DTYPE_USED}")
    return model, processor, device


## 7. Segmentação SAM 3 e extração SHPM

As funções abaixo reutilizam a versão corrigida do notebook-base: `torch.inference_mode()`, autocast em CUDA, `bfloat16` quando suportado, fallback para `float16`, prompts genéricos para olhos, separação horizontal dos olhos, remoção de duplicatas por IoU e registro de avisos por score baixo.


In [ ]:
def tensor_or_array_to_numpy_mask(mask: Any) -> np.ndarray:
    if isinstance(mask, torch.Tensor):
        arr = mask.detach().cpu().numpy()
    else:
        arr = np.asarray(mask)

    arr = np.squeeze(arr)
    if arr.ndim != 2:
        raise ValueError(f"Máscara com dimensão inesperada: {arr.shape}")

    if arr.dtype != bool:
        arr = arr > 0.5

    return arr.astype(bool)


def sam3_segment_parts(
    img_rgb: Image.Image,
    processor: Any,
    score_threshold: float = SCORE_THRESHOLD,
) -> tuple[dict[str, np.ndarray], dict[str, float], list[str]]:
    width, height = img_rgb.size
    empty_mask = np.zeros((height, width), dtype=bool)

    part_masks: dict[str, np.ndarray] = {}
    part_scores: dict[str, float] = {}
    part_errors: list[str] = []

    if torch.cuda.is_available():
        supports_bfloat16 = getattr(torch.cuda, "is_bf16_supported", lambda: False)()
        amp_dtype = torch.bfloat16 if supports_bfloat16 else torch.float16
        inference_context = torch.autocast(device_type="cuda", dtype=amp_dtype)
    else:
        inference_context = nullcontext()

    def run_prompt(inference_state: Any, prompt: str) -> list[dict[str, Any]]:
        output = processor.set_text_prompt(state=inference_state, prompt=prompt)

        masks = output.get("masks")
        scores = output.get("scores")

        if masks is None or scores is None or len(masks) == 0:
            return []

        scores_tensor = torch.as_tensor(scores).reshape(-1)
        if scores_tensor.numel() == 0:
            return []

        candidate_count = min(len(masks), scores_tensor.numel())
        candidates: list[dict[str, Any]] = []

        for index in range(candidate_count):
            mask = tensor_or_array_to_numpy_mask(masks[index])

            if mask.shape != (height, width):
                continue

            area = int(mask.sum())
            if area <= 0:
                continue

            y_coordinates, x_coordinates = np.where(mask)
            candidates.append({
                "mask": mask,
                "score": float(scores_tensor[index].item()),
                "area": area,
                "center_x": float(x_coordinates.mean()),
                "center_y": float(y_coordinates.mean()),
            })

        return candidates

    def mask_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
        intersection = np.logical_and(mask_a, mask_b).sum()
        union = np.logical_or(mask_a, mask_b).sum()
        if union == 0:
            return 0.0
        return float(intersection / union)

    def remove_duplicate_candidates(candidates: list[dict[str, Any]], maximum_iou: float = 0.50) -> list[dict[str, Any]]:
        ordered = sorted(candidates, key=lambda candidate: candidate["score"], reverse=True)
        unique: list[dict[str, Any]] = []

        for candidate in ordered:
            is_duplicate = any(
                mask_iou(candidate["mask"], saved["mask"]) > maximum_iou
                for saved in unique
            )
            if not is_duplicate:
                unique.append(candidate)

        return unique

    with torch.inference_mode(), inference_context:
        inference_state = processor.set_image(img_rgb)

        non_eye_parts = ["face", "nose", "mouth", "hair"]
        for part_name in non_eye_parts:
            prompt = SAM3_PROMPTS[part_name]
            try:
                candidates = run_prompt(inference_state=inference_state, prompt=prompt)

                if not candidates:
                    part_masks[part_name] = empty_mask.copy()
                    part_scores[part_name] = np.nan
                    part_errors.append(f"{part_name}: nenhuma máscara retornada pelo SAM 3")
                    continue

                best_candidate = max(candidates, key=lambda candidate: candidate["score"])
                best_score = best_candidate["score"]

                part_masks[part_name] = best_candidate["mask"]
                part_scores[part_name] = best_score

                if best_score < score_threshold:
                    part_errors.append(
                        f"AVISO {part_name}: melhor score {best_score:.3f} abaixo do limiar {score_threshold:.3f}"
                    )

            except Exception as exc:
                part_masks[part_name] = empty_mask.copy()
                part_scores[part_name] = np.nan
                part_errors.append(f"{part_name}: {type(exc).__name__}: {exc}")

        try:
            eye_candidates: list[dict[str, Any]] = []
            for eye_prompt in ["eye", "human eye"]:
                eye_candidates.extend(run_prompt(inference_state=inference_state, prompt=eye_prompt))

            eye_candidates = remove_duplicate_candidates(eye_candidates)
            eye_candidates = sorted(eye_candidates, key=lambda candidate: candidate["score"], reverse=True)[:2]

            if len(eye_candidates) < 2:
                part_masks["l_eye"] = empty_mask.copy()
                part_masks["r_eye"] = empty_mask.copy()
                part_scores["l_eye"] = np.nan
                part_scores["r_eye"] = np.nan
                part_errors.append("eyes: não foi possível encontrar duas máscaras distintas de olhos")
            else:
                eye_candidates = sorted(eye_candidates, key=lambda candidate: candidate["center_x"])
                image_left_eye = eye_candidates[0]
                image_right_eye = eye_candidates[1]

                part_masks["r_eye"] = image_left_eye["mask"]
                part_scores["r_eye"] = image_left_eye["score"]
                part_masks["l_eye"] = image_right_eye["mask"]
                part_scores["l_eye"] = image_right_eye["score"]

                for part_name in ["l_eye", "r_eye"]:
                    score = part_scores[part_name]
                    if score < score_threshold:
                        part_errors.append(
                            f"AVISO {part_name}: melhor score {score:.3f} abaixo do limiar {score_threshold:.3f}"
                        )

        except Exception as exc:
            part_masks["l_eye"] = empty_mask.copy()
            part_masks["r_eye"] = empty_mask.copy()
            part_scores["l_eye"] = np.nan
            part_scores["r_eye"] = np.nan
            part_errors.append(f"eyes: {type(exc).__name__}: {exc}")

    for part_name in parts:
        part_masks.setdefault(part_name, empty_mask.copy())
        part_scores.setdefault(part_name, np.nan)

    return part_masks, part_scores, part_errors


def clean_mask(mask: np.ndarray) -> np.ndarray:
    mask = np.asarray(mask, dtype=bool)
    opened = binary_opening(mask, structure=np.ones((3, 3), dtype=bool))
    closed = binary_closing(opened, structure=np.ones((3, 3), dtype=bool))
    return closed.astype(bool)


def clean_masks(raw_masks: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    return {part: clean_mask(raw_masks.get(part, np.zeros((1, 1), dtype=bool))) for part in parts}


def summarize_mask_areas(masks: dict[str, np.ndarray]) -> dict[str, int]:
    return {part: int(np.asarray(masks[part], dtype=bool).sum()) for part in parts}


def validate_masks(areas: dict[str, int], part_errors: list[str], min_face_area: int = MIN_FACE_AREA) -> None:
    fatal_errors = [error for error in part_errors if not error.startswith("AVISO ")]

    if fatal_errors:
        raise ValueError("; ".join(fatal_errors))

    face_area = areas.get("face", 0)
    if face_area < min_face_area:
        raise ValueError(f"face inválida: área {face_area} menor que MIN_FACE_AREA={min_face_area}")

    missing_parts = [part for part in parts if areas.get(part, 0) <= 0]
    if missing_parts:
        raise ValueError(f"regiões essenciais ausentes após limpeza: {missing_parts}")


def extract_shpm_features_from_areas(areas: dict[str, int]) -> dict[str, float]:
    values: dict[str, float] = {}

    for part_i in parts:
        for part_j in parts:
            if part_i == part_j:
                continue
            area_i = float(areas.get(part_i, 0))
            area_j = float(areas.get(part_j, 0))
            values[f"{part_i}_vs_{part_j}"] = area_i / area_j if area_j > 0 else 0.0

    assert list(values.keys()) == feature_names
    assert len(values) == 30
    return values


## 8. Controle de qualidade visual



In [ ]:
def safe_stem(path: str | Path) -> str:
    stem = Path(path).stem
    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", stem).strip("._")
    return stem or "image"


def save_binary_masks(mask_dir: Path, masks: dict[str, np.ndarray]) -> None:
    mask_dir.mkdir(parents=True, exist_ok=True)
    for part, mask in masks.items():
        Image.fromarray((np.asarray(mask, dtype=np.uint8) * 255), mode="L").save(mask_dir / f"{part}.png")


def save_combined_overlay(output_path: Path, img_rgb: Image.Image, masks: dict[str, np.ndarray], alpha: float = 0.42) -> None:
    image_arr = np.asarray(img_rgb.convert("RGB"), dtype=float)
    overlay = image_arr.copy()

    for part, mask in masks.items():
        color = np.asarray(MASK_COLORS[part], dtype=float)
        mask_bool = np.asarray(mask, dtype=bool)
        overlay[mask_bool] = (1.0 - alpha) * overlay[mask_bool] + alpha * color

    Image.fromarray(np.clip(overlay, 0, 255).astype(np.uint8)).save(output_path)


def save_segmentation_panel(
    output_path: Path,
    img_rgb: Image.Image,
    masks: dict[str, np.ndarray],
    areas: dict[str, int],
    scores: dict[str, float],
    title: str,
) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(12, 8), dpi=150)
    axes = axes.ravel()
    base = np.asarray(img_rgb.convert("RGB"))

    for ax, part in zip(axes, parts):
        ax.imshow(base)
        color = np.asarray(MASK_COLORS[part], dtype=float) / 255.0
        colored = np.zeros((*masks[part].shape, 4), dtype=float)
        colored[..., :3] = color
        colored[..., 3] = np.asarray(masks[part], dtype=bool) * 0.55
        ax.imshow(colored)
        score = scores.get(part, np.nan)
        score_text = "nan" if not np.isfinite(score) else f"{score:.3f}"
        ax.set_title(f"{part}\nárea={areas.get(part, 0)} | score={score_text}", fontsize=10)
        ax.axis("off")

    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(output_path, dpi=220)
    plt.close(fig)


def save_qc_outputs(
    qc_dir: Path,
    relative_path: str,
    img_rgb: Image.Image,
    masks: dict[str, np.ndarray],
    areas: dict[str, int],
    scores: dict[str, float],
) -> None:
    image_stem = safe_stem(relative_path)
    qc_dir.mkdir(parents=True, exist_ok=True)
    img_rgb.save(qc_dir / f"{image_stem}_original.jpg")
    save_combined_overlay(qc_dir / f"{image_stem}_overlay.png", img_rgb, masks)
    save_segmentation_panel(qc_dir / f"{image_stem}_panel.png", img_rgb, masks, areas, scores, relative_path)
    save_binary_masks(qc_dir / f"{image_stem}_masks", masks)


## 9. Checkpoints e retomada real



In [ ]:
PROCESSED_CHECKPOINT = CHECKPOINT_ROOT / "processed_rows.parquet"
FAILURES_CHECKPOINT = CHECKPOINT_ROOT / "failures.parquet"
PROCESSED_CSV_FALLBACK = CHECKPOINT_ROOT / "processed_rows.csv"
FAILURES_CSV_FALLBACK = CHECKPOINT_ROOT / "failures.csv"


def atomic_write_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp_path, index=False)
    tmp_path.replace(path)


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)


def load_checkpoint(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()


def checkpoint_save(processed_rows: list[dict[str, Any]], failure_rows: list[dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    processed_df = pd.DataFrame(processed_rows)
    failures_df = pd.DataFrame(failure_rows)

    if not processed_df.empty:
        processed_df = processed_df.drop_duplicates("relative_path", keep="last").reset_index(drop=True)
    if not failures_df.empty:
        failures_df = failures_df.drop_duplicates("relative_path", keep="last").reset_index(drop=True)

    atomic_write_parquet(processed_df, PROCESSED_CHECKPOINT)
    atomic_write_csv(processed_df, PROCESSED_CSV_FALLBACK)
    atomic_write_parquet(failures_df, FAILURES_CHECKPOINT)
    atomic_write_csv(failures_df, FAILURES_CSV_FALLBACK)

    return processed_df, failures_df


existing_processed_df = load_checkpoint(PROCESSED_CHECKPOINT)
existing_failures_df = load_checkpoint(FAILURES_CHECKPOINT)

existing_processed_rows = existing_processed_df.to_dict("records") if not existing_processed_df.empty else []
existing_failure_rows = existing_failures_df.to_dict("records") if not existing_failures_df.empty else []
already_processed = set(existing_processed_df["relative_path"].astype(str)) if "relative_path" in existing_processed_df.columns else set()

remaining_metadata_df = selected_metadata_df.loc[
    ~selected_metadata_df["relative_path"].astype(str).isin(already_processed)
].reset_index(drop=True)

print(f"Registros já processados: {len(already_processed)}")
print(f"Registros restantes nesta execução: {len(remaining_metadata_df)}")


## 10. Processamento das imagens



In [ ]:
AREA_COLUMNS = [f"area_{part}" for part in parts]
AREA_ALIAS_COLUMNS = [f"{part}_area" for part in parts]
SCORE_COLUMNS = [f"{part}_score" for part in parts]

METADATA_COLUMNS = [
    "dataset",
    "filename",
    "relative_path",
    "gender",
    "celeb_name",
    "celeb_id",
    "photo_taken",
    "dob",
    "status",
    "error_message",
]

# Estas três métricas são realmente auxiliares.
# hair_vs_face, nose_vs_face e mouth_vs_face já pertencem a feature_names.
AUXILIARY_COLUMNS = [
    "eyes_total_area",
    "eyes_total_vs_face",
    "eye_asymmetry",
]

# dict.fromkeys preserva a ordem e impede nomes duplicados.
METRICS_COLUMNS = list(dict.fromkeys(
    METADATA_COLUMNS
    + AREA_COLUMNS
    + AREA_ALIAS_COLUMNS
    + SCORE_COLUMNS
    + feature_names
    + AUXILIARY_COLUMNS
    + [
        "segmentation_warnings",
        "metadata_face_score",
        "metadata_second_face_score",
        "image_path",
    ]
))

assert len(METRICS_COLUMNS) == len(set(METRICS_COLUMNS))


def empty_metric_row(meta: pd.Series, status: str, error_message: str) -> dict[str, Any]:
    row = {column: np.nan for column in METRICS_COLUMNS}
    row.update({
        "dataset": meta["dataset"],
        "filename": meta["filename"],
        "relative_path": meta["relative_path"],
        "gender": meta["gender"],
        "celeb_name": meta.get("celeb_name", ""),
        "celeb_id": meta.get("celeb_id", np.nan),
        "photo_taken": meta.get("photo_taken", np.nan),
        "dob": meta.get("dob", np.nan),
        "status": status,
        "error_message": error_message,
        "metadata_face_score": meta.get("metadata_face_score", np.nan),
        "metadata_second_face_score": meta.get("metadata_second_face_score", np.nan),
        "image_path": meta.get("image_path", ""),
    })
    return row


def auxiliary_features(areas: dict[str, int]) -> dict[str, float]:
    l_eye = float(areas.get("l_eye", 0))
    r_eye = float(areas.get("r_eye", 0))
    face = float(areas.get("face", 0))
    eyes_total = l_eye + r_eye
    eye_denominator = max(eyes_total, 1.0)

    return {
        "eyes_total_area": eyes_total,
        "eyes_total_vs_face": eyes_total / face if face > 0 else 0.0,
        "eye_asymmetry": abs(l_eye - r_eye) / eye_denominator,
    }


def should_save_qc(meta: pd.Series, status: str, qc_counts: dict[str, int], failure_count: int) -> tuple[bool, Path, int]:
    if not SAVE_QC_IMAGES:
        return False, QC_ROOT, failure_count

    if status == "valid":
        gender = str(meta["gender"])
        if qc_counts.get(gender, 0) < QC_IMAGES_PER_GENDER:
            qc_counts[gender] = qc_counts.get(gender, 0) + 1
            return True, QC_ROOT / "valid" / gender, failure_count
        return False, QC_ROOT, failure_count

    if failure_count < QC_FAILURE_IMAGES:
        failure_count += 1
        return True, QC_ROOT / "failures", failure_count

    return False, QC_ROOT, failure_count


def process_image_row(
    meta: pd.Series,
    processor: Any,
    qc_counts: dict[str, int],
    failure_count: int,
) -> tuple[dict[str, Any], dict[str, Any] | None, int]:
    try:
        image_path = Path(meta["image_path"])
        img_rgb = Image.open(image_path).convert("RGB")

        raw_masks, scores, part_errors = sam3_segment_parts(img_rgb, processor, SCORE_THRESHOLD)
        masks = clean_masks(raw_masks)
        areas = summarize_mask_areas(masks)
        validate_masks(areas, part_errors, MIN_FACE_AREA)

        shpm_values = extract_shpm_features_from_areas(areas)
        aux_values = auxiliary_features(areas)

        row = empty_metric_row(meta, status="valid", error_message="")
        for part in parts:
            row[f"area_{part}"] = areas[part]
            row[f"{part}_area"] = areas[part]
            row[f"{part}_score"] = scores.get(part, np.nan)

        row.update(shpm_values)
        row.update(aux_values)
        row["segmentation_warnings"] = "; ".join([error for error in part_errors if error.startswith("AVISO ")])

        save_qc, qc_dir, failure_count = should_save_qc(meta, "valid", qc_counts, failure_count)
        if save_qc:
            save_qc_outputs(qc_dir, meta["relative_path"], img_rgb, masks, areas, scores)

        del raw_masks, masks, img_rgb
        return row, None, failure_count

    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"
        row = empty_metric_row(meta, status="invalid", error_message=message)

        try:
            image_path = Path(meta["image_path"])
            if image_path.exists():
                img_rgb = Image.open(image_path).convert("RGB")
                width, height = img_rgb.size
                empty = np.zeros((height, width), dtype=bool)
                masks = {part: empty for part in parts}
                areas = {part: 0 for part in parts}
                scores = {part: np.nan for part in parts}
                save_qc, qc_dir, failure_count = should_save_qc(meta, "invalid", qc_counts, failure_count)
                if save_qc:
                    save_qc_outputs(qc_dir, meta["relative_path"], img_rgb, masks, areas, scores)
                del masks, img_rgb
        except Exception as qc_exc:
            row["segmentation_warnings"] = f"falha ao salvar QC da falha: {type(qc_exc).__name__}: {qc_exc}"

        return row, row.copy(), failure_count


model, processor, device = initialize_sam3()

processed_rows = list(existing_processed_rows)
failure_rows = list(existing_failure_rows)
qc_counts = {"Female": 0, "Male": 0}
failure_qc_count = 0

for position, (_, meta) in enumerate(tqdm(remaining_metadata_df.iterrows(), total=len(remaining_metadata_df), desc="Processando"), start=1):
    row, failure_row, failure_qc_count = process_image_row(meta, processor, qc_counts, failure_qc_count)
    processed_rows.append(row)
    if failure_row is not None:
        failure_rows.append(failure_row)

    if position % CHECKPOINT_EVERY == 0:
        checkpoint_df, checkpoint_failures_df = checkpoint_save(processed_rows, failure_rows)
        valid_count = int((checkpoint_df["status"] == "valid").sum()) if not checkpoint_df.empty else 0
        invalid_count = int((checkpoint_df["status"] != "valid").sum()) if not checkpoint_df.empty else 0
        print(f"Checkpoint {position}: válidas={valid_count} | inválidas={invalid_count}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

final_checkpoint_df, final_failures_checkpoint_df = checkpoint_save(processed_rows, failure_rows)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Processamento concluído.")
print(final_checkpoint_df["status"].value_counts(dropna=False) if not final_checkpoint_df.empty else "Sem linhas processadas.")


## 11. Exportação dos CSVs e Parquet




In [ ]:
def finite_feature_mask(df: pd.DataFrame) -> pd.Series:
    feature_matrix = df[feature_names].apply(pd.to_numeric, errors="coerce")
    return feature_matrix.notna().all(axis=1) & np.isfinite(feature_matrix.to_numpy(dtype=float)).all(axis=1)


def build_reference_valid(all_df: pd.DataFrame) -> pd.DataFrame:
    if all_df.empty:
        return pd.DataFrame(columns=["filename", "gender", "area_face", *feature_names])

    valid = all_df.loc[
        (all_df["status"] == "valid")
        & all_df["gender"].isin(["Female", "Male"])
        & finite_feature_mask(all_df)
    ].copy()

    valid = valid.loc[(valid[AREA_COLUMNS] > 0).all(axis=1)].copy()
    reference_columns = ["filename", "gender", "area_face", *feature_names]
    return valid[reference_columns].reset_index(drop=True)


def build_reference_balanced(reference_valid_df: pd.DataFrame) -> pd.DataFrame:
    if reference_valid_df.empty:
        return reference_valid_df.copy()

    counts = reference_valid_df["gender"].value_counts()
    n = min(int(counts.get("Female", 0)), int(counts.get("Male", 0)))
    if n == 0:
        return reference_valid_df.iloc[0:0].copy()

    parts_df = []
    for gender in ["Female", "Male"]:
        group = reference_valid_df.loc[reference_valid_df["gender"] == gender].sort_values("filename")
        parts_df.append(group.sample(n=n, random_state=RANDOM_SEED))
    return pd.concat(parts_df, axis=0).sort_values(["gender", "filename"]).reset_index(drop=True)


all_metrics_df = final_checkpoint_df.copy()

# Proteção para checkpoints ou DataFrames gerados por versões anteriores.
all_metrics_df = all_metrics_df.loc[:, ~all_metrics_df.columns.duplicated()].copy()
export_columns = list(dict.fromkeys(METRICS_COLUMNS))

if not all_metrics_df.empty:
    for column in export_columns:
        if column not in all_metrics_df.columns:
            all_metrics_df[column] = np.nan
    all_metrics_df = (
        all_metrics_df[export_columns]
        .drop_duplicates("relative_path", keep="last")
        .reset_index(drop=True)
    )

failures_df = all_metrics_df.loc[all_metrics_df["status"] != "valid"].copy() if not all_metrics_df.empty else pd.DataFrame(columns=METRICS_COLUMNS)
reference_valid_df = build_reference_valid(all_metrics_df)
reference_balanced_df = build_reference_balanced(reference_valid_df)

metrics_all_csv = OUTPUT_ROOT / f"{FILE_PREFIX}_shpm_metrics_all.csv"
reference_valid_csv = OUTPUT_ROOT / f"{FILE_PREFIX}_shpm_reference_valid.csv"
reference_balanced_csv = OUTPUT_ROOT / f"{FILE_PREFIX}_shpm_reference_balanced.csv"
failures_csv = OUTPUT_ROOT / f"{FILE_PREFIX}_shpm_failures.csv"
metrics_all_parquet = OUTPUT_ROOT / f"{FILE_PREFIX}_shpm_metrics_all.parquet"
manifest_json = OUTPUT_ROOT / f"{FILE_PREFIX}_shpm_manifest.json"

all_metrics_df.to_csv(metrics_all_csv, index=False)
reference_valid_df.to_csv(reference_valid_csv, index=False)
reference_balanced_df.to_csv(reference_balanced_csv, index=False)
failures_df.to_csv(failures_csv, index=False)
all_metrics_df.to_parquet(metrics_all_parquet, index=False)

print("Arquivos exportados:")
for path in [metrics_all_csv, reference_valid_csv, reference_balanced_csv, failures_csv, metrics_all_parquet]:
    print(path.resolve(), path.exists(), human_size(path.stat().st_size) if path.exists() else "missing")


## 12. Validações obrigatórias e relatórios finais


In [ ]:
EXPECTED_REFERENCE_COLUMNS = ["filename", "gender", "area_face", *feature_names]

assert len(feature_names) == 30
assert set(METADATA_COLUMNS).issubset(all_metrics_df.columns)
assert set(AREA_COLUMNS).issubset(all_metrics_df.columns)
assert set(AREA_ALIAS_COLUMNS).issubset(all_metrics_df.columns)
assert set(SCORE_COLUMNS).issubset(all_metrics_df.columns)
assert set(feature_names).issubset(all_metrics_df.columns)
assert list(reference_balanced_df.columns) == EXPECTED_REFERENCE_COLUMNS
assert SAM3_CHECKPOINT == "sam3.pt"
assert get_sam3_code_commit() == SAM3_CODE_COMMIT

if not all_metrics_df.empty:
    assert not all_metrics_df["relative_path"].duplicated().any()

if not reference_valid_df.empty:
    assert set(reference_valid_df["gender"].unique()).issubset({"Female", "Male"})
    assert (reference_valid_df[feature_names].notna().all(axis=1)).all()
    assert np.isfinite(reference_valid_df[feature_names].to_numpy(dtype=float)).all()
    valid_full_rows = all_metrics_df.loc[all_metrics_df["status"] == "valid"]
    assert (valid_full_rows[AREA_COLUMNS] > 0).all().all()

if not reference_balanced_df.empty:
    balanced_counts = reference_balanced_df["gender"].value_counts()
    assert balanced_counts.get("Female", 0) == balanced_counts.get("Male", 0)

success_rate = 0.0 if all_metrics_df.empty else float((all_metrics_df["status"] == "valid").mean())

print("Shapes:")
print({
    "all_metrics_df": all_metrics_df.shape,
    "reference_valid_df": reference_valid_df.shape,
    "reference_balanced_df": reference_balanced_df.shape,
    "failures_df": failures_df.shape,
})

print("\nContagem por gender no CSV válido:")
print(reference_valid_df["gender"].value_counts() if not reference_valid_df.empty else "Sem válidos")

print(f"\nTaxa de sucesso: {success_rate:.2%}")

print("\nPrincipais causas de falha:")
if failures_df.empty:
    print("Sem falhas registradas.")
else:
    display(failures_df["error_message"].fillna("").str.slice(0, 160).value_counts().head(20))

print("\nEstatística descritiva das seis áreas:")
if not all_metrics_df.empty:
    display(all_metrics_df.loc[all_metrics_df["status"] == "valid", AREA_COLUMNS].describe())
else:
    display(pd.DataFrame(columns=AREA_COLUMNS))

print("\nEstatística descritiva dos atributos auxiliares independentes:")
if not all_metrics_df.empty:
    display(all_metrics_df.loc[all_metrics_df["status"] == "valid", AUXILIARY_COLUMNS].describe())
else:
    display(pd.DataFrame(columns=AUXILIARY_COLUMNS))

print("\nUso aproximado de disco:")
disk_report(OUTPUT_ROOT)

print("\nCaminhos absolutos dos arquivos finais:")
for path in [metrics_all_csv, reference_valid_csv, reference_balanced_csv, failures_csv, metrics_all_parquet, manifest_json]:
    print(path.resolve())


## 13. Gráficos descritivos de controle de qualidade



In [ ]:
def save_summary_plots(all_df: pd.DataFrame, valid_df: pd.DataFrame) -> None:
    SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

    if all_df.empty:
        print("Sem dados para gráficos.")
        return

    fig, ax = plt.subplots(figsize=(5, 4), dpi=160)
    valid_df["gender"].value_counts().reindex(["Female", "Male"]).fillna(0).plot(kind="bar", ax=ax, color=["#d9a7c7", "#93b7dc"])
    ax.set_title("Contagem Female/Male em amostras válidas")
    ax.set_ylabel("N")
    fig.tight_layout()
    fig.savefig(SUMMARY_ROOT / "gender_counts.png", dpi=220)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5, 4), dpi=160)
    all_df["status"].value_counts().plot(kind="bar", ax=ax, color=["#9ecae1", "#fdae6b"])
    ax.set_title("Taxa válida/inválida")
    ax.set_ylabel("N")
    fig.tight_layout()
    fig.savefig(SUMMARY_ROOT / "valid_invalid_counts.png", dpi=220)
    plt.close(fig)

    for feature in ["nose_vs_face", "mouth_vs_face", "eyes_total_vs_face"]:
        fig, ax = plt.subplots(figsize=(6, 4), dpi=160)
        if sns is not None and not valid_df.empty:
            sns.histplot(data=valid_df, x=feature, hue="gender", bins=40, element="step", stat="count", common_norm=False, ax=ax)
        else:
            for gender, color in [("Female", "#d9a7c7"), ("Male", "#93b7dc")]:
                subset = valid_df.loc[valid_df["gender"] == gender, feature].dropna()
                ax.hist(subset, bins=40, alpha=0.55, label=gender, color=color)
            ax.legend()
        ax.set_title(f"Histograma de {feature}")
        fig.tight_layout()
        fig.savefig(SUMMARY_ROOT / f"hist_{feature}.png", dpi=220)
        plt.close(fig)

    box_features = ["nose_vs_face", "mouth_vs_face", "eyes_total_vs_face"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=160)
    for ax, feature in zip(axes, box_features):
        if sns is not None and not valid_df.empty:
            sns.boxplot(data=valid_df, x="gender", y=feature, order=["Female", "Male"], ax=ax)
        else:
            data = [
                valid_df.loc[valid_df["gender"] == "Female", feature].dropna(),
                valid_df.loc[valid_df["gender"] == "Male", feature].dropna(),
            ]
            ax.boxplot(data, labels=["Female", "Male"])
        ax.set_title(feature)
    fig.tight_layout()
    fig.savefig(SUMMARY_ROOT / "boxplots_auxiliary_features_by_gender.png", dpi=220)
    plt.close(fig)

    print(f"Gráficos salvos em: {SUMMARY_ROOT.resolve()}")


valid_for_plots_df = all_metrics_df.loc[all_metrics_df["status"] == "valid"].copy() if not all_metrics_df.empty else pd.DataFrame()
save_summary_plots(all_metrics_df, valid_for_plots_df)


## 14. Manifesto de reprodutibilidade


In [ ]:
def package_version(module: Any) -> str | None:
    return getattr(module, "__version__", None)


manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "DATASET_KIND": DATASET_KIND,
    "dataset_url": DATASET_URL,
    "archive_path": str(ARCHIVE_PATH),
    "archive_size_bytes": ARCHIVE_PATH.stat().st_size if ARCHIVE_PATH.exists() else None,
    "image_count_selected": int(len(selected_metadata_df)),
    "image_count_processed": int(len(all_metrics_df)),
    "valid_count": int((all_metrics_df["status"] == "valid").sum()) if not all_metrics_df.empty else 0,
    "invalid_count": int((all_metrics_df["status"] != "valid").sum()) if not all_metrics_df.empty else 0,
    "female_count_valid": int((reference_valid_df["gender"] == "Female").sum()) if not reference_valid_df.empty else 0,
    "male_count_valid": int((reference_valid_df["gender"] == "Male").sum()) if not reference_valid_df.empty else 0,
    "RUN_MODE": RUN_MODE,
    "MAX_IMAGES_PER_GENDER": MAX_IMAGES_PER_GENDER,
    "RANDOM_SEED": RANDOM_SEED,
    "SCORE_THRESHOLD": SCORE_THRESHOLD,
    "MIN_FACE_AREA": MIN_FACE_AREA,
    "SAM3_PROMPTS": SAM3_PROMPTS,
    "parts": parts,
    "feature_names": feature_names,
    "SAM3_CODE_COMMIT": SAM3_CODE_COMMIT,
    "SAM3_MODEL_REPO": SAM3_MODEL_REPO,
    "SAM3_CHECKPOINT": SAM3_CHECKPOINT,
    "SAM3_CHECKPOINT_PATH": str(SAM3_CHECKPOINT_PATH) if SAM3_CHECKPOINT_PATH is not None else None,
    "hf_revision": HF_REVISION,
    "python_version": sys.version,
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "torchvision_version": package_version(torchvision) if torchvision is not None else None,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "autocast_dtype": AUTOCast_DTYPE_USED,
    "metadata_filter_report": metadata_filter_report,
    "output_files": {
        "metrics_all_csv": str(metrics_all_csv.resolve()),
        "reference_valid_csv": str(reference_valid_csv.resolve()),
        "reference_balanced_csv": str(reference_balanced_csv.resolve()),
        "failures_csv": str(failures_csv.resolve()),
        "metrics_all_parquet": str(metrics_all_parquet.resolve()),
        "manifest_json": str(manifest_json.resolve()),
    },
}

with open(manifest_json, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"Manifesto salvo em: {manifest_json.resolve()}")
display(pd.DataFrame([{
    "processed": manifest["image_count_processed"],
    "valid": manifest["valid_count"],
    "invalid": manifest["invalid_count"],
    "female_valid": manifest["female_count_valid"],
    "male_valid": manifest["male_count_valid"],
    "run_mode": RUN_MODE,
    "dataset": DATASET_KIND,
}]))
